<h1 style='text-align: center;'>Titanic survival prediction</h1>

The sinking of the Titanic is one of the most infamous shipwrecks in history.

On April 15, 1912, during her maiden voyage, the widely considered “unsinkable” RMS Titanic sank after colliding with an iceberg. Unfortunately, there weren’t enough lifeboats for everyone onboard, resulting in the death of 1502 out of 2224 passengers and crew.

While there was some element of luck involved in surviving, it seems some groups of people were more likely to survive than others.

In this challenge, we ask you to build a predictive model that answers the question: “what sorts of people were more likely to survive?” using passenger data (ie name, age, gender, socio-economic class, etc).

<h2>Variable Notes</h2>

pclass: A proxy for socio-economic status (SES)
1st = Upper
2nd = Middle
3rd = Lower

age: Age is fractional if less than 1. If the age is estimated, is it in the form of xx.5

sibsp: The dataset defines family relations in this way...
Sibling = brother, sister, stepbrother, stepsister
Spouse = husband, wife (mistresses and fiancés were ignored)

parch: The dataset defines family relations in this way...
Parent = mother, father
Child = daughter, son, stepdaughter, stepson
Some children travelled only with a nanny, therefore parch=0 for them.

In [1]:
import pandas as pd
import os
from pathlib import Path

For the first step we need to import all the datasets, one for training and one for testing correspondingly:

In [2]:
training_data = pd.read_csv('train.csv')
test_data = pd.read_csv('test.csv')

In [3]:
training_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


In [4]:
test_data.head()

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,892,3,"Kelly, Mr. James",male,34.5,0,0,330911,7.8292,NaN,Q
1,893,3,"Wilkes, Mrs. James (Ellen Needs)",female,47.0,1,0,363272,7.0000,NaN,S
2,894,2,"Myles, Mr. Thomas Francis",male,62.0,0,0,240276,9.6875,NaN,Q
3,895,3,"Wirz, Mr. Albert",male,27.0,0,0,315154,8.6625,NaN,S
4,896,3,"Hirvonen, Mrs. Alexander (Helga E Lindqvist)",female,22.0,1,1,3101298,12.2875,NaN,S


In [5]:
training_data.describe()


,PassengerId,Survived,Pclass,Age,SibSp,Parch,Fare
count,891.000000,891.000000,891.000000,714.000000,891.000000,891.000000,891.000000
mean,446.000000,0.383838,2.308642,29.699118,0.523008,0.381594,32.204208
std,257.353842,0.486592,0.836071,14.526497,1.102743,0.806057,49.693429
min,1.000000,0.000000,1.000000,0.420000,0.000000,0.000000,0.000000
25%,223.500000,0.000000,2.000000,20.125000,0.000000,0.000000,7.910400
50%,446.000000,0.000000,3.000000,28.000000,0.000000,0.000000,14.454200
75%,668.500000,1.000000,3.000000,38.000000,1.000000,0.000000,31.000000
max,891.000000,1.000000,3.000000,80.000000,8.000000,6.000000,512.329200


In first place, let's try to find the most important people on the ship - Jack Dawson and Rose DeWitt Bukater, main characters of same name movie.
To do that we supposed to somehow filter the 'Name' column:

In [6]:
training_data[training_data['Name'].str.contains(
    r'dowson|dewitt|rose|bukater',
    case=False,
    na=False
)]

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
72,73,0,2,"Hood, Mr. Ambrose Jr",male,21.0,0,0,S.O.C. 14879,73.50,NaN,S
855,856,1,3,"Aks, Mrs. Sam (Leah Rosen)",female,18.0,0,1,392091,9.35,NaN,S


In [7]:
test_data[test_data['Name'].str.contains(
    'dawson|dewitt|rose|bukater|jacK',
    case=False,
    na=False
)]

,PassengerId,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
208,1100,1,"Rosenbaum, Miss. Edith Louise",female,33.0,0,0,PC 17613,27.7208,A11,C
327,1219,1,"Rosenshine, Mr. George (Mr George Thorne"")""",male,46.0,0,0,PC 17585,79.2000,NaN,C


Unfortunately, we found out that characters are fictional and we should move on to the next task. Now it's time to take actions with data, but we need to choose obvious features we will work with which are not noises or pure identificators. Therefore we gonnna refuse all columns which are not really answering the question:  “what sorts of people were more likely to survive?”.

Here are there:
1. Id
2. Name
3. Ticket
4. Cabin (arguably, will be checked later)

They don't give any helpful data so they are out of the discussion from now on, so here is the target_features variable sure without of our y


In [8]:
training_data.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='str')

In [9]:
target_features = ['Pclass', 'Sex', 'Age', 'SibSp',
       'Parch', 'Fare', 'Embarked']

In [10]:
training_data.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


Time to make some preprocessing, but firstly we need to divide data in target (survived) and other features to make proper data imputation without data leakage

In [11]:
X = training_data[target_features]
y = training_data['Survived']
print(y)
X

0      0
1      1
2      1
3      1
4      0
      ..
886    0
887    1
888    0
889    1
890    0
Name: Survived, Length: 891, dtype: int64


,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S
...,...,...,...,...,...,...,...
886,2,male,27.0,0,0,13.0000,S
887,1,female,19.0,0,0,30.0000,S
888,3,female,NaN,1,2,23.4500,S
889,1,male,26.0,0,0,30.0000,C


Data is divided, next stage is train test split:

In [12]:
from sklearn.model_selection import train_test_split

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

In [13]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,
    random_state=42,
    stratify=y
)

Time to analize if dataset has missing values:

In [14]:
print('X:\n',X.isna().sum()[lambda x: x > 0], sep='')
print('\ny:\n',y.isna().sum(), sep='')

X:
Age         177
Embarked      2
dtype: int64

y:
0


Beforehand the beginning of the data imputation we need to divide the features into categorical, numerical and ordinal the choos proper method for each.

(FOR BOTH DATASETS)
!!!

As the data shows, we have 4 missing values columns which are: Age, Cabin, Embarked, Fair.

Division:
1. Categorical: Embarked, Cabin
2. Ordinal: none
3. Numerical: Age, Fair

All of them are quite significant in terms of the question of the research: “what sorts of people were more likely to survive?”, so every piece should be handled. So for every category we need to create a pipeline. 

!!!

FOR TRAINING_DATA:

Division:
1. Categorical: Embarked, Sex, 
2. Ordinal: Pclass, 
3. Numerical: Age, Fair, Parch, SibSp

Categorical data has imputations so we use Simple Imputer and for encoding data we use OneHotEncoder:

In [15]:
cat_pipe = Pipeline ([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore'))
])

Numerical data also also uses Simple Imputer but with strategy for numbers and scales data with StandardScaler. But in this case it's not that simple cause we don't really know what will work out better - median or mean, so we need to check out skew of our numericals:

In [16]:
X.head()

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
0,3,male,22.0,1,0,7.2500,S
1,1,female,38.0,1,0,71.2833,C
2,3,female,26.0,0,0,7.9250,S
3,1,female,35.0,1,0,53.1000,S
4,3,male,35.0,0,0,8.0500,S


In [17]:
X[['Age', 'Fare']].skew()

Age     0.389108
Fare    4.787317
dtype: float64

As we can see our data in fare especially has a long right tail so data divided not evenly so better to use Median, but age is opposite so it's arguable, so we can check out the difference between median and mean:

In [18]:
print('Median:', X[['Age', 'Fare']].median())
print('Mean:', X[['Age', 'Fare']].mean())

Median: Age     28.0000
Fare    14.4542
dtype: float64
Mean: Age     29.699118
Fare    32.204208
dtype: float64


Median and Mean are almost equal, the closest to the 50 is the best but in this case it's negligable so for first check we'll leave it this way, in case we won't get requiered accuracy it's always possible to create another pipeline for mean.

In [19]:
num_pipe = Pipeline ([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

Pipelines are done, time to create preprocessor with transformer:

In [20]:
preprocessor = ColumnTransformer(
    transformers =[
        ('cat',cat_pipe, ['Embarked', 'Sex', 'Pclass']),
        ('num', num_pipe, ['Age', 'Fare', 'Parch', 'SibSp'])
    ]
)

Preprocessor is done, now we can easily transform data without missing data due to imputation, but we still didn't train a model yet, so now it's time to create models for LogisticRegression and RandoForestClassifier:

In [21]:
lr_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(random_state=42))
])

rfc_pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', RandomForestClassifier(random_state=42))
])

Recommended next step is to check how the data transformed:

In [22]:

import numpy as np

final_X = preprocessor.fit_transform(X_train)
np.isnan(final_X).sum()

np.int64(0)

As we can see there are no nan so we handled data. Eventually we've done with the models, now the only thing until evaluation left is training:

In [23]:
lr_pipe.fit(X_train, y_train);

In [24]:
rfc_pipe.fit(X_train, y_train);

Time for validation:

In [25]:
lr_pred = lr_pipe.predict(X_test)
rfc_pred = rfc_pipe.predict(X_test)

In [26]:
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

We will begin from Linear regression, continue with tree and finish evaluation with final comparison:

In [27]:
print('Accuracy:\n', accuracy_score(y_test, lr_pred))
print('Confustion matrix:\n', confusion_matrix(y_test, lr_pred))
print('Classification report:\n', classification_report(y_test, lr_pred))

Accuracy:
 0.7985074626865671
Confustion matrix:
 [[142  23]
 [ 31  72]]
Classification report:
               precision    recall  f1-score   support

           0       0.82      0.86      0.84       165
           1       0.76      0.70      0.73       103

    accuracy                           0.80       268
   macro avg       0.79      0.78      0.78       268
weighted avg       0.80      0.80      0.80       268



In [28]:
print('Accuracy:\n', accuracy_score(y_test, rfc_pred))
print('Confustion matrix:\n', confusion_matrix(y_test, rfc_pred))
print('Classification report:\n', classification_report(y_test, rfc_pred))

Accuracy:
 0.8022388059701493
Confustion matrix:
 [[144  21]
 [ 32  71]]
Classification report:
               precision    recall  f1-score   support

           0       0.82      0.87      0.84       165
           1       0.77      0.69      0.73       103

    accuracy                           0.80       268
   macro avg       0.79      0.78      0.79       268
weighted avg       0.80      0.80      0.80       268



The validation test shows that both model coped with a task quite good but in total RFC was a bit better in terms of accuracy but a little worse in coverage, 0.70 for survived in LR and 0.69 in RFC respectively. Without any addinional changes (adding Cabin feature and changing meadian to mean for Age in imputer, that's all I noticed for now) it's believed to me that next step is using full train data set and testing it on the kaggle test data (which also requires some changes):

Currently we are done but we need to use cross-validation to understand average metric for the whole dataset and exclude chances that accuracy was reached by a blind luck

In [29]:
from sklearn.model_selection import StratifiedKFold, cross_val_score

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_lr = cross_val_score(lr_pipe, X_train, y_train, cv=skf, scoring='accuracy')
cv_rfc = cross_val_score(rfc_pipe, X_train, y_train, cv=skf, scoring='accuracy')

print(f"Logistic Regression 5-Fold CV: {cv_lr.mean():.2f} (± {cv_lr.std():.4f})")
print(f"Random Forest 5-Fold CV:       {cv_rfc.mean():.2f} (± {cv_rfc.std():.4f})")

Logistic Regression 5-Fold CV: 0.80 (± 0.0311)
Random Forest 5-Fold CV:       0.81 (± 0.0253)


The results showed us that accuracy for both models is actual and stable for the 5 different stratified folds, so we can continue

In [30]:
final_rfc = rfc_pipe.fit(X, y)

Test dataset analysis and imputation:

In [31]:
print('\nTest data:\n',test_data.isna().sum().loc[lambda x: x > 0], sep='')


Test data:
Age       86
Fare       1
Cabin    327
dtype: int64


In [33]:
predicted = final_rfc.predict(test_data[target_features])

submission = pd.DataFrame({
    'PassengerId':test_data['PassengerId'],
    'Survived':predicted
})

submission.to_csv('submission.csv', index=False)

<h1 style='text-align: center;'>Titanic survival prediction</h1>

For now it's the end of research, after all the things done project covered all the requiered topics except of cross-validation (now acknowladged with it yet). Two different models were implemented and tested. RFC model got pretty high accuracy 81% which is 4 higher than requiered. The answer on the research question is not answered yet because requiers a lot of additional analysis, but the goal was to finish ML pipeline which is finished.  

Next goal: 
1. Answer the question and see the dependencies
2. Prove guesses with the data

The cross-validation with Kstratified was added above proving the accuracy of the models, but so far the dataset wasn't really analized, we only made predictions so it's time to change it. Let's start from discovering the weight of the every feature we used:

In [67]:
coeffs = lr_pipe.named_steps['model'].coef_[0]
feature_names = lr_pipe.named_steps['preprocessor'].get_feature_names_out()

rfc_importance = rfc_pipe.named_steps['model'].feature_importances_
rfc_feature_names = rfc_pipe.named_steps['preprocessor'].get_feature_names_out()

lr_df = pd.DataFrame({
    'Feature name': feature_names,
    'Coefficients': coeffs
}).sort_values(by='Coefficients', ascending=False, key=abs).reset_index(drop=True)

rfc_df = pd.DataFrame({
    'Feature name': rfc_feature_names,
    'Importance': rfc_importance
}).sort_values(by='Importance', ascending=False).reset_index(drop=True)

pd.concat([lr_df, rfc_df], axis=1)

,Feature name,Coefficients,Feature name,Importance
0,cat__Sex_female,1.359233,num__Age,0.253982
1,cat__Sex_male,-1.244768,num__Fare,0.252166
2,cat__Pclass_1,1.031665,cat__Sex_male,0.154474
3,cat__Pclass_3,-1.027470,cat__Sex_female,0.126351
4,num__Age,-0.536012,cat__Pclass_3,0.056469
5,cat__Embarked_Q,0.318782,num__SibSp,0.046996
6,num__SibSp,-0.295234,num__Parch,0.036969
7,cat__Embarked_S,-0.266376,cat__Pclass_1,0.025160
8,num__Fare,0.145442,cat__Pclass_2,0.014439
9,cat__Pclass_2,0.110270,cat__Embarked_S,0.012812


The model analisys presented that both have quite different top-3 most important factors

Let's examine the model on the total amount of mistakes:

In [94]:
mistakes = X_test.copy()
mistakes['Truth'] = y_test
mistakes['Prediction'] = rfc_pred

errors = mistakes[mistakes['Truth']!= mistakes['Prediction']]

print(f'Total amount of errors is {len(errors)}')
print(f'Classes 1 and 3 amount of errors: {len(errors[errors['Pclass'].isin([1, 3])])}')
print(f'Classes 1: {len(errors[errors['Pclass']==1])} and 3: {len(errors[errors['Pclass']==3])}')

Total amount of errors is 53
Classes 1 and 3 amount of errors: 50
Classes 1: 15 and 3: 35


We can see that most of the mistakes happened on the edge values of classses, 1 and 3 correspondingly. Ration of mistakes bigger on 3, need to check why:

In [86]:
errors[errors['Pclass'].isin([ 3])]

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked,Truth,Prediction
804,3,male,27.0,0,0,6.9750,S,1,0
338,3,male,45.0,0,0,8.0500,S,1,0
431,3,female,NaN,1,0,16.1000,S,1,0
205,3,female,2.0,0,1,10.4625,S,0,1
489,3,male,9.0,1,1,15.9000,S,1,0
868,3,male,NaN,0,0,9.5000,S,0,1
347,3,female,NaN,1,0,16.1000,S,1,0
442,3,male,25.0,1,0,7.7750,S,0,1
826,3,male,NaN,0,0,56.4958,S,0,1
444,3,male,NaN,0,0,8.1125,S,1,0


In [95]:
pd.crosstab(errors[errors['Pclass']==3]['Sex'], errors[errors['Pclass']==3]['Truth'])

Truth,0,1
Sex,,
female,7,11
male,6,11


Results show us that tree too dependable on age and sex which is not always truth and was wrong mostly in this cases. Female, young, should survive and male, young or dead but due to 3 class probably died. Model requires tuning and feature engeneering too improve most of the edge cases and discover additional patterns. But there are so many cases where objects suited to every criteria still were predicted as wrong.

Post-mortem:
Target and Scope
Binary classification task predicting Titanic passenger survival using 891 training records and 418 test records.

Feature Selection
Included primary signals: Pclass, Sex, Age, SibSp, Parch, Fare, and Embarked. Excluded PassengerId, Name, Ticket, and Cabin to avoid overfitting and noise.

Leakage Prevention and Validation
All imputation, one-hot encoding, and standard scaling steps were encapsulated in ColumnTransformer inside Pipeline and fitted on training data only. A 5-fold Stratified K-Fold validation protocol with random seed 42 was applied.

Model Comparison and Selection
Both models surpassed the minimum validation threshold of 0.780. Logistic Regression achieved 0.80 average cross-validation accuracy, and Random Forest achieved 0.81. Random Forest was selected as the final model for the 418-row submission file due to higher accuracy and lower variance.

Interpretability and Error Analysis
Logistic Regression identified female sex (+1.36) and first class (+1.03) as the strongest positive factors, and male sex (-1.24) and third class (-1.03) as the strongest negative factors. Random Forest placed the highest importance on age (25.4%) and fare (25.2%). Out of 53 errors on holdout data, 50 occurred in first and third class, with 35 in third class where the model overestimated gender survival rules and failed to capture physical barriers to lifeboats.